
# Ejercicio C - Cover de "La Vaca Lola" con animación sincronizada

**Objetivo:** realizar una producción multimedia de la canción infantil usando un audio propio y una animación de monigotes/vaca sincronizada con el ritmo.

Este notebook permite:

1. Subir un archivo de audio del cover (`.mp3`, `.wav`, `.m4a`).
2. Analizar el ritmo del audio mediante energía RMS y detección de beats.
3. Generar una animación 2D estilo escenario musical.
4. Sincronizar el movimiento de la vaca/monigote con el ritmo.
5. Exportar un video `.mp4` listo para presentar.

> Importante: el notebook **no incluye audio ni letra** de la canción. Debes subir tu propio cover grabado o autorizado.



## 1. Instalación de librerías
Ejecuta esta celda primero. Colab instalará las dependencias necesarias para analizar audio y renderizar video.


In [ ]:

!pip -q install librosa moviepy==1.0.3 imageio-ffmpeg pillow numpy matplotlib



## 2. Importar librerías


In [ ]:

import os
import math
import json
import zipfile
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from google.colab import files
from moviepy.editor import VideoClip, AudioFileClip
from IPython.display import Video, display



## 3. Subir audio del cover
Sube tu archivo de audio. Puede ser `.mp3`, `.wav`, `.m4a` u otro formato compatible.

Recomendación: para que el renderizado sea rápido, usa un audio de 30 a 90 segundos.


In [ ]:

uploaded = files.upload()

if len(uploaded) == 0:
    raise Exception('No se subió ningún archivo de audio.')

audio_path = list(uploaded.keys())[0]
print('Audio cargado:', audio_path)



## 4. Configuración del proyecto multimedia
Puedes cambiar el nombre del artista de referencia y el estilo visual.

Para evitar clonación de voz o uso indebido de imagen, se utiliza solo una **inspiración estética general**.


In [ ]:

# Configuración editable
TITULO_PROYECTO = 'Cover de La Vaca Lola'
ARTISTA_REFERENCIA = 'Artista pop conocido - referencia visual'
ESTILO_VISUAL = 'Escenario pop infantil con luces y ritmo'

# Duración máxima en segundos para renderizar. Puedes subirlo si tu computadora/Colab aguanta.
DURACION_MAXIMA = 75
FPS = 24
ANCHO = 1280
ALTO = 720

print('Proyecto:', TITULO_PROYECTO)
print('Referencia visual:', ARTISTA_REFERENCIA)
print('Estilo:', ESTILO_VISUAL)



## 5. Analizar ritmo del audio
Se calcula:

- **RMS:** energía del sonido en cada instante.
- **Tempo:** velocidad aproximada del audio.
- **Beats:** golpes de ritmo detectados.

Estos datos controlan el tamaño, salto, movimiento y luces de la animación.


In [ ]:

y, sr = librosa.load(audio_path, sr=22050, mono=True)
duracion_original = librosa.get_duration(y=y, sr=sr)
duracion = min(duracion_original, DURACION_MAXIMA)

# Recortar si es muy largo para evitar render demasiado lento
y = y[:int(duracion * sr)]

hop_length = 512
rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop_length)[0]
rms = rms / (np.max(rms) + 1e-9)
times_rms = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop_length)

tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr, hop_length=hop_length)
beat_times = librosa.frames_to_time(beat_frames, sr=sr, hop_length=hop_length)

print('Duración original:', round(duracion_original, 2), 's')
print('Duración renderizada:', round(duracion, 2), 's')
print('Tempo aproximado:', round(float(tempo), 2), 'BPM')
print('Cantidad de beats detectados:', len(beat_times))



## 6. Visualización del análisis de audio
Esta gráfica sirve como evidencia para el informe: se ve cómo el sistema detecta la energía del audio y los beats usados para sincronizar la animación.


In [ ]:

plt.figure(figsize=(14, 4))
plt.plot(times_rms, rms, label='Energía RMS')
for bt in beat_times:
    if bt <= duracion:
        plt.axvline(bt, alpha=0.25)
plt.title('Análisis de ritmo del audio: energía y beats')
plt.xlabel('Tiempo (segundos)')
plt.ylabel('Energía normalizada')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('analisis_ritmo_vaca_lola.png', dpi=150)
plt.show()



## 7. Funciones para dibujar la animación
La animación se genera cuadro por cuadro. El movimiento cambia según la energía del audio y la cercanía al beat.


In [ ]:

def obtener_energia(t):
    idx = np.searchsorted(times_rms, t)
    idx = min(max(idx, 0), len(rms) - 1)
    return float(rms[idx])


def cercania_beat(t):
    if len(beat_times) == 0:
        return 0.0
    d = np.min(np.abs(beat_times - t))
    return max(0.0, 1.0 - d / 0.18)


def fuente(size=32, bold=False):
    rutas = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf' if bold else '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf' if bold else '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf'
    ]
    for r in rutas:
        if os.path.exists(r):
            return ImageFont.truetype(r, size)
    return ImageFont.load_default()


def texto_centrado(draw, xy, text, font, fill=(255,255,255)):
    x, y = xy
    bbox = draw.textbbox((0, 0), text, font=font)
    w = bbox[2] - bbox[0]
    draw.text((x - w / 2, y), text, font=font, fill=fill)


def dibujar_vaca(draw, cx, cy, escala=1.0, boca_abierta=False, paso=0.0):
    s = escala
    # sombra
    draw.ellipse((cx-170*s, cy+125*s, cx+170*s, cy+165*s), fill=(20,20,30,120))
    # cuerpo
    draw.ellipse((cx-160*s, cy-60*s, cx+160*s, cy+110*s), fill=(250,250,245), outline=(35,35,45), width=max(2,int(4*s)))
    # manchas
    draw.ellipse((cx-100*s, cy-30*s, cx-45*s, cy+35*s), fill=(25,25,25))
    draw.ellipse((cx+40*s, cy-40*s, cx+105*s, cy+35*s), fill=(25,25,25))
    draw.ellipse((cx-15*s, cy+40*s, cx+45*s, cy+90*s), fill=(25,25,25))
    # patas animadas
    leg_offset = math.sin(paso) * 12 * s
    for lx in [-95, -35, 45, 105]:
        draw.rounded_rectangle((cx+(lx-12)*s, cy+80*s+leg_offset, cx+(lx+12)*s, cy+165*s+leg_offset), radius=int(10*s), fill=(245,245,240), outline=(35,35,45), width=max(2,int(3*s)))
        draw.rectangle((cx+(lx-15)*s, cy+150*s+leg_offset, cx+(lx+15)*s, cy+170*s+leg_offset), fill=(30,30,30))
        leg_offset *= -1
    # cola
    draw.line((cx+150*s, cy+15*s, cx+210*s, cy-40*s + math.sin(paso)*20*s), fill=(35,35,45), width=max(3,int(5*s)))
    draw.ellipse((cx+200*s, cy-55*s + math.sin(paso)*20*s, cx+230*s, cy-25*s + math.sin(paso)*20*s), fill=(25,25,25))
    # cabeza
    hx, hy = cx-105*s, cy-95*s
    draw.ellipse((hx-85*s, hy-70*s, hx+85*s, hy+80*s), fill=(250,250,245), outline=(35,35,45), width=max(2,int(4*s)))
    # orejas
    draw.ellipse((hx-115*s, hy-45*s, hx-65*s, hy+15*s), fill=(245,218,220), outline=(35,35,45), width=max(2,int(3*s)))
    draw.ellipse((hx+65*s, hy-45*s, hx+115*s, hy+15*s), fill=(245,218,220), outline=(35,35,45), width=max(2,int(3*s)))
    # cuernos
    draw.polygon([(hx-45*s,hy-60*s),(hx-25*s,hy-120*s),(hx-5*s,hy-60*s)], fill=(235,215,155), outline=(35,35,45))
    draw.polygon([(hx+45*s,hy-60*s),(hx+25*s,hy-120*s),(hx+5*s,hy-60*s)], fill=(235,215,155), outline=(35,35,45))
    # manchas cabeza
    draw.ellipse((hx-55*s, hy-25*s, hx-10*s, hy+20*s), fill=(25,25,25))
    # ojos
    draw.ellipse((hx-35*s, hy-10*s, hx-15*s, hy+15*s), fill=(255,255,255), outline=(0,0,0))
    draw.ellipse((hx+15*s, hy-10*s, hx+35*s, hy+15*s), fill=(255,255,255), outline=(0,0,0))
    draw.ellipse((hx-26*s, hy-1*s, hx-18*s, hy+8*s), fill=(0,0,0))
    draw.ellipse((hx+24*s, hy-1*s, hx+32*s, hy+8*s), fill=(0,0,0))
    # hocico
    draw.ellipse((hx-50*s, hy+25*s, hx+50*s, hy+85*s), fill=(245,190,195), outline=(35,35,45), width=max(2,int(3*s)))
    draw.ellipse((hx-25*s, hy+48*s, hx-10*s, hy+62*s), fill=(40,40,40))
    draw.ellipse((hx+10*s, hy+48*s, hx+25*s, hy+62*s), fill=(40,40,40))
    if boca_abierta:
        draw.ellipse((hx-20*s, hy+65*s, hx+20*s, hy+90*s), fill=(70,20,30))
    else:
        draw.arc((hx-25*s, hy+55*s, hx+25*s, hy+82*s), 15, 165, fill=(70,20,30), width=max(2,int(3*s)))


def dibujar_frame(t):
    energia = obtener_energia(t)
    beat = cercania_beat(t)
    pulso = 0.5 + 0.5 * math.sin(2 * math.pi * 1.2 * t)
    salto = (energia * 45 + beat * 55) * math.sin(2 * math.pi * (float(tempo)/60.0 if tempo else 1.6) * t)
    escala = 1.0 + 0.12 * beat + 0.06 * energia
    paso = 2 * math.pi * t * (float(tempo)/60.0 if tempo else 1.8)

    img = Image.new('RGB', (ANCHO, ALTO), (10, 12, 28))
    draw = ImageDraw.Draw(img)

    # fondo degradado
    for ypix in range(ALTO):
        r = int(12 + 35 * (ypix / ALTO))
        g = int(14 + 10 * (ypix / ALTO))
        b = int(40 + 70 * (ypix / ALTO))
        draw.line((0, ypix, ANCHO, ypix), fill=(r,g,b))

    # luces del escenario sincronizadas
    colores = [(255, 218, 80), (120, 220, 255), (255, 110, 200), (130, 255, 150)]
    for i, x in enumerate([120, 400, 880, 1160]):
        intensity = int(70 + 120 * (energia + beat) / 2)
        c = colores[i]
        fill = tuple(min(255, int(v * (0.35 + 0.65*(energia+beat)/2))) for v in c)
        draw.polygon([(x, 60), (x-220, 520), (x+220, 520)], fill=fill)
        draw.ellipse((x-28, 35, x+28, 91), fill=c, outline=(255,255,255))

    # escenario
    draw.rectangle((0, 520, ANCHO, ALTO), fill=(25, 22, 42))
    draw.ellipse((-100, 480, ANCHO+100, 800), fill=(42, 40, 75))
    draw.line((0, 520, ANCHO, 520), fill=(255,255,255), width=3)

    # barras de ecualizador
    for i in range(32):
        bx = 40 + i * 38
        altura = int(25 + 150 * (energia * (0.4 + 0.6 * abs(math.sin(t*3+i)))))
        draw.rounded_rectangle((bx, 500-altura, bx+20, 500), radius=5, fill=(80, 220, 200))

    # vaca principal
    dibujar_vaca(draw, ANCHO//2, int(405 - abs(salto)), escala=escala, boca_abierta=energia>0.38 or beat>0.5, paso=paso)

    # monigotes bailarines
    for i, mx in enumerate([220, 1060]):
        my = 430 + int(math.sin(paso + i) * 15)
        color = (255, 235, 90) if i == 0 else (140, 210, 255)
        draw.ellipse((mx-30, my-120, mx+30, my-60), fill=color, outline=(0,0,0), width=3)
        draw.line((mx, my-60, mx, my+30), fill=color, width=10)
        arm = math.sin(paso*1.2 + i) * 50
        draw.line((mx, my-25, mx-60, my-80-arm), fill=color, width=8)
        draw.line((mx, my-25, mx+60, my-80+arm), fill=color, width=8)
        draw.line((mx, my+30, mx-45, my+100), fill=color, width=8)
        draw.line((mx, my+30, mx+45, my+100), fill=color, width=8)

    # títulos
    f_title = fuente(48, True)
    f_sub = fuente(25, False)
    f_small = fuente(21, False)
    texto_centrado(draw, (ANCHO//2, 28), TITULO_PROYECTO, f_title, fill=(255,255,255))
    texto_centrado(draw, (ANCHO//2, 86), ESTILO_VISUAL, f_sub, fill=(190,220,255))
    texto_centrado(draw, (ANCHO//2, 645), 'Animación sincronizada con energía RMS y beats del audio', f_small, fill=(210,240,235))

    # indicador de beat
    radius = int(20 + beat * 35)
    draw.ellipse((ANCHO-95-radius, 65-radius, ANCHO-95+radius, 65+radius), outline=(255,255,255), width=4)
    draw.text((ANCHO-165, 115), 'BEAT', font=fuente(18, True), fill=(255,255,255))

    return np.array(img)



## 8. Vista previa de un fotograma
Antes de renderizar todo el video, revisa cómo se verá la animación.


In [ ]:

preview = dibujar_frame(min(3, duracion/2))
plt.figure(figsize=(12, 7))
plt.imshow(preview)
plt.axis('off')
plt.title('Vista previa de la animación')
plt.show()



## 9. Generar video sincronizado
Esta celda crea el `.mp4`. Puede tardar varios minutos según la duración del audio.


In [ ]:

video_sin_audio = VideoClip(lambda t: dibujar_frame(t), duration=duracion)
clip_audio = AudioFileClip(audio_path).subclip(0, duracion)
video_final = video_sin_audio.set_audio(clip_audio)

salida_video = 'cover_vaca_lola_animado.mp4'
video_final.write_videofile(
    salida_video,
    fps=FPS,
    codec='libx264',
    audio_codec='aac',
    bitrate='2500k',
    preset='medium',
    verbose=False,
    logger='bar'
)

print('Video generado:', salida_video)



## 10. Reproducir resultado


In [ ]:

display(Video(salida_video, embed=True, width=900))



## 11. Guardar resumen técnico para el informe
Se genera un JSON con datos técnicos del audio y del video.


In [ ]:

resumen = {
    'ejercicio': 'C - Cover de La Vaca Lola',
    'tipo_produccion': 'Audio + animacion 2D sincronizada',
    'audio_usado': audio_path,
    'duracion_original_segundos': round(float(duracion_original), 2),
    'duracion_renderizada_segundos': round(float(duracion), 2),
    'tempo_bpm_aproximado': round(float(tempo), 2),
    'beats_detectados': int(len(beat_times)),
    'fps': FPS,
    'resolucion': f'{ANCHO}x{ALTO}',
    'metodo_sincronizacion': 'Energia RMS + deteccion de beats con librosa',
    'archivo_video': salida_video,
    'archivo_grafica': 'analisis_ritmo_vaca_lola.png'
}

with open('resumen_tecnico_vaca_lola.json', 'w', encoding='utf-8') as f:
    json.dump(resumen, f, ensure_ascii=False, indent=4)

resumen



## 12. Comprimir resultados y descargar
El ZIP final contiene:

- Video `.mp4`
- Gráfica del análisis de ritmo `.png`
- Resumen técnico `.json`


In [ ]:

zip_salida = 'resultados_cover_vaca_lola.zip'
with zipfile.ZipFile(zip_salida, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(salida_video)
    z.write('analisis_ritmo_vaca_lola.png')
    z.write('resumen_tecnico_vaca_lola.json')

print('ZIP creado:', zip_salida)
files.download(zip_salida)



## Texto breve para el informe

**Metodología utilizada:**  
Se desarrolló una producción multimedia basada en un audio propio del cover de la canción infantil. El audio fue analizado mediante energía RMS y detección de beats. Con estos valores, el sistema generó una animación 2D cuadro por cuadro, haciendo que el personaje principal y los elementos visuales del escenario se muevan de acuerdo con la intensidad y ritmo del sonido.

**Herramientas y tecnologías:**  
Google Colab, Python, Librosa para análisis de audio, Pillow para dibujo de fotogramas y MoviePy para la exportación del video final en formato MP4.

**Resultado obtenido:**  
Se obtuvo un video animado sincronizado con el cover de audio, donde la vaca, los monigotes y las luces reaccionan al ritmo detectado. Además, se generó una gráfica de energía RMS y beats como evidencia técnica del proceso de sincronización.
